# AI Job Trends Dashboard: Digital/AI Demand vs. Supply

## Portfolio Project Overview

This project builds an interactive country-level dashboard to analyze where digital and AI labor-market demand appears to be rising faster than workforce supply. It uses public development and labor indicators to create a practical, reproducible framework for identifying talent gaps, workforce bottlenecks, and areas where digital transformation may outpace labor-market readiness.

### Problem framing
Leaders need better signals to understand where digital and AI adoption is accelerating and whether labor markets are prepared to support that growth. Because globally consistent AI job-posting microdata is not always openly available, this project uses a proxy-based approach built from publicly accessible country-level indicators.

### Why this matters
As digitalization and AI reshape work, countries and organizations need ways to assess whether talent pipelines, connectivity, and digital employment capacity are keeping pace. A dashboard like this can support workforce planning, economic development strategy, skills investment, and cross-country benchmarking.


## Systems engineering approach (requirements → design)

### What “demand vs supply” means in this project
Because globally consistent AI job posting microdata is not always openly downloadable, this notebook uses a practical, reproducible approach:

**Digital/AI demand index (country-year):** a composite of:
- Growth in ICT services exports (proxy for digital economy pull)
- Growth in high-technology exports (proxy for tech-intensive production)
- Internet adoption and broadband penetration (proxy for market readiness to absorb digital work)

**Digital/AI supply index (country-year):** a composite of:
- ICT employment (ICT services + ICT manufacturing)
- Tertiary enrollment (proxy for skills pipeline)

### Dashboard views
1) Gap map: countries colored by (Demand Index − Supply Index)  
2) Quadrant plot: Demand vs Supply (“Priority upskilling”, “Scale demand”, etc.)  
3) Trend drill-down: per-country trajectories (2015–latest)  
4) Export HTML: shareable single-file dashboard

### Output artifacts
- Clean merged dataset (CSV)
- Plotly dashboard exported to HTML
- Written story + recommendations


In [16]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
import plotly.express as px

pd.set_option("display.max_columns", 200)


## Scope (chosen to be globally balanced)
High-income: USA, GBR, DEU, KOR  
Upper-middle: CHN, BRA, MEX  
Lower-middle: IND, VNM, NGA  


In [17]:
countries = {
    "USA": "United States",
    "GBR": "United Kingdom",
    "DEU": "Germany",
    "KOR": "Korea, Rep.",
    "CHN": "China",
    "BRA": "Brazil",
    "MEX": "Mexico",
    "IND": "India",
    "VNM": "Vietnam",
    "NGA": "Nigeria",
}

START_YEAR = 2015
END_YEAR = 2024


## Step 1 — Pull WDI indicators (connectivity + digital economy demand proxies)

Indicators:
- IT.NET.USER.ZS Internet users (% of population)  
- IT.NET.BBND.P2 Fixed broadband (per 100 people)  
- BX.GSR.CCIS.ZS ICT service exports (% of service exports)  
- TX.VAL.TECH.CD High-technology exports (US$)  
- SE.TER.ENRR Tertiary enrollment (% gross)  
- SP.POP.TOTL Population (for per-capita conversion)  


In [18]:
WDI_BASE = "https://api.worldbank.org/v2"

wdi_indicators = {
    "internet_users_pct": "IT.NET.USER.ZS",
    "fixed_broadband_per100": "IT.NET.BBND.P2",
    "ict_service_exports_pct": "BX.GSR.CCIS.ZS",
    "high_tech_exports_usd": "TX.VAL.TECH.CD",
    "tertiary_enroll_gross_pct": "SE.TER.ENRR",
    "population": "SP.POP.TOTL",
}

def fetch_wdi(country_iso3: str, indicator: str) -> pd.DataFrame:
    url = f"{WDI_BASE}/country/{country_iso3}/indicator/{indicator}"
    params = {"format": "json", "per_page": 20000}
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    rows = []
    for obs in data[1] if len(data) > 1 and data[1] else []:
        year = int(obs["date"])
        if START_YEAR <= year <= END_YEAR:
            rows.append({"iso3": country_iso3, "year": year, "value": obs["value"]})
    return pd.DataFrame(rows)

def build_wdi_panel() -> pd.DataFrame:
    # Initialize a list to hold all indicator DataFrames, each with data for all countries
    all_indicator_dfs = []

    for col, ind in wdi_indicators.items():
        # Fetch data for this specific indicator for all countries
        indicator_data_for_all_countries = []
        for iso3 in countries.keys():
            df_single_country_single_indicator = fetch_wdi(iso3, ind).rename(columns={"value": col})
            if not df_single_country_single_indicator.empty:
                indicator_data_for_all_countries.append(df_single_country_single_indicator)

        if indicator_data_for_all_countries:
            # Concatenate all single-country dataframes for this indicator vertically
            # This results in a DataFrame with iso3, year, and the current indicator column for all countries
            df_current_indicator = pd.concat(indicator_data_for_all_countries, ignore_index=True)
            all_indicator_dfs.append(df_current_indicator)

    # Now, merge all these indicator-specific dataframes into a single wide panel
    panel = None
    if all_indicator_dfs:
        panel = all_indicator_dfs[0]
        for i in range(1, len(all_indicator_dfs)):
            # Here, each merge adds a new indicator column,
            # because the 'on' columns are 'iso3' and 'year', and the other column is a unique indicator.
            panel = panel.merge(all_indicator_dfs[i], on=["iso3","year"], how="outer")
    else:
        # If no data is fetched at all, return an empty dataframe with expected columns
        return pd.DataFrame(columns=["iso3", "year"] + list(wdi_indicators.keys()) + ["country"])

    panel["country"] = panel["iso3"].map(countries)
    return panel.sort_values(["iso3","year"]).reset_index(drop=True)

wdi = build_wdi_panel()
wdi.head(10)


/tmp/ipykernel_416/1996902737.py:40: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,iso3,year,internet_users_pct,fixed_broadband_per100,ict_service_exports_pct,high_tech_exports_usd,tertiary_enroll_gross_pct,population,country
0,BRA,2015,58.3280,12.6354,4.666642,9.433129e+09,48.434040,201675532,Brazil
1,BRA,2016,60.8725,13.2177,5.457031,1.037554e+10,48.333340,203218114,Brazil
2,BRA,2017,67.4713,14.1218,6.573564,1.071520e+10,50.026749,204703445,Brazil
3,BRA,2018,70.4343,15.1538,7.688418,1.106319e+10,51.191689,206107261,Brazil
4,BRA,2019,73.9124,15.8622,7.791551,9.392110e+09,52.624062,207455459,Brazil
5,BRA,2020,81.3427,17.4181,9.274362,5.944952e+09,54.572811,208660842,Brazil
6,BRA,2021,80.6899,19.8363,10.483914,6.350129e+09,56.830620,209550294,Brazil
7,BRA,2022,80.5278,20.9534,11.601448,7.707210e+09,60.390621,210306415,Brazil
8,BRA,2023,84.1506,22.4192,12.745838,8.060842e+09,NaN,211140729,Brazil
9,BRA,2024,84.4635,NaN,12.853044,8.694668e+09,NaN,211998573,Brazil


## Step 2 — Pull ICT employment (Data360 / ILO_EMP)

If the direct URLs work in your environment:
- https://data360files.worldbank.org/data360-data/data/ILO_EMP/ILO_EMP_ICT_MAN.csv  
- https://data360files.worldbank.org/data360-data/data/ILO_EMP/ILO_EMP_ICT_SER.csv  

If not, download from Data360 manually and load locally.


In [19]:
def fetch_csv(url: str) -> pd.DataFrame:
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    return pd.read_csv(StringIO(r.text))

# Original URLs for ILO_EMP_ICT_MAN.csv and ILO_EMP_ICT_SER.csv are no longer working (404 Client Error).
# We are now using World Bank Development Indicators (WDI) as programmatic proxies for ICT employment data.
# Note: These WDI indicators are percentages of total employment, not raw numbers, and
# 'ict_mfg_emp' uses a general manufacturing employment indicator due to lack of a
# specific ICT manufacturing employment indicator in WDI.

# World Bank Development Indicators (WDI) for employment proxies
WDI_ICT_SRV_EMP_INDICATOR = "IS.ICT.EMPL.ZS"  # Employment in ICT services (% of total employment)
WDI_MANF_EMP_INDICATOR = "IS.IND.MANF.ZS"    # Employment in manufacturing (% of total employment) - Proxy for ICT manufacturing employment

# To ensure compatibility with the `tidy_ict` function in the next cell,
# we need to fetch data for all countries for each indicator and concatenate them.
# The `fetch_wdi` function (defined in cell 5187d5c3) already returns data in a format
# that `tidy_ict` can process (iso3, year, value).

all_ict_man_data = []
all_ict_ser_data = []

for iso3 in countries.keys():
    # Fetch data for ICT services employment
    df_srv = fetch_wdi(iso3, WDI_ICT_SRV_EMP_INDICATOR)
    if not df_srv.empty:
        all_ict_ser_data.append(df_srv)

    # Fetch data for manufacturing employment (proxy for ICT manufacturing)
    df_mfg = fetch_wdi(iso3, WDI_MANF_EMP_INDICATOR)
    if not df_mfg.empty:
        all_ict_man_data.append(df_mfg)

# Concatenate all dataframes for each "raw" type
ict_man = pd.concat(all_ict_man_data, ignore_index=True) if all_ict_man_data else pd.DataFrame(columns=["iso3", "year", "value"])
ict_ser = pd.concat(all_ict_ser_data, ignore_index=True) if all_ict_ser_data else pd.DataFrame(columns=["iso3", "year", "value"])

ict_man.head()

,iso3,year,value


## Step 3 — Clean, merge, and compute indices

Demand Index (mean z-score):
- ICT service exports %, High-tech exports per capita, Internet users %, Broadband per 100

Supply Index (mean z-score):
- ICT employment (services + manufacturing), Tertiary enrollment

Gap Index = Demand − Supply


In [20]:
df = wdi.copy()

def tidy_ict(df_raw: pd.DataFrame, value_name: str) -> pd.DataFrame:
    cols = {c.lower(): c for c in df_raw.columns}
    # Ensure 'iso3' is picked up for economy code
    econ = cols.get("iso3") or cols.get("economy code") or cols.get("economy_code") or cols.get("economycode") or cols.get("country code")
    time = cols.get("time") or cols.get("year")
    val = cols.get("value") or cols.get("data") or cols.get("obs_value")

    # Check if any of the identified column names are None. If so, it means the expected columns are not present
    # or the dataframe is truly empty without the expected 'iso3', 'year', 'value' initialisation.
    # Given `ict_man` and `ict_ser` are explicitly created with 'iso3', 'year', 'value' columns even if empty,
    # econ, time, and val should not be None here.
    # This ensures it works even for an empty DataFrame structured as `pd.DataFrame(columns=['iso3', 'year', 'value'])`
    out = df_raw[[econ, time, val]].copy()
    out.columns = ["iso3", "year", value_name]
    out["year"] = pd.to_numeric(out["year"], errors="coerce")
    out[value_name] = pd.to_numeric(out[value_name], errors="coerce")
    out = out.dropna(subset=["iso3","year"])
    out["year"] = out["year"].astype(int)
    return out[(out["year"] >= START_YEAR) & (out["year"] <= END_YEAR)]

ict_man_t = tidy_ict(ict_man, "ict_mfg_emp")
ict_ser_t = tidy_ict(ict_ser, "ict_srv_emp")

df = df.merge(ict_man_t, on=["iso3","year"], how="left")
df = df.merge(ict_ser_t, on=["iso3","year"], how="left")

df["high_tech_exports_per_capita"] = df["high_tech_exports_usd"] / df["population"]

def zscore(s: pd.Series) -> pd.Series:
    mu = s.mean(skipna=True)
    sd = s.std(skipna=True)
    if sd is None or np.isnan(sd) or sd == 0:
        return s * np.nan
    return (s - mu) / sd

demand_components = ["ict_service_exports_pct","high_tech_exports_per_capita","internet_users_pct","fixed_broadband_per100"]
supply_components = ["ict_srv_emp","ict_mfg_emp","tertiary_enroll_gross_pct"]

for c in demand_components + supply_components:
    df[f"z_{c}"] = zscore(df[c])

df["demand_index"] = df[[f"z_{c}" for c in demand_components]].mean(axis=1, skipna=True)
df["supply_index"] = df[[f"z_{c}" for c in supply_components]].mean(axis=1, skipna=True)
df["gap_index"] = df["demand_index"] - df["supply_index"]

df.head(10)


,iso3,year,internet_users_pct,fixed_broadband_per100,ict_service_exports_pct,high_tech_exports_usd,tertiary_enroll_gross_pct,population,country,ict_mfg_emp,ict_srv_emp,high_tech_exports_per_capita,z_ict_service_exports_pct,z_high_tech_exports_per_capita,z_internet_users_pct,z_fixed_broadband_per100,z_ict_srv_emp,z_ict_mfg_emp,z_tertiary_enroll_gross_pct,demand_index,supply_index,gap_index
0,BRA,2015,58.3280,12.6354,4.666642,9.433129e+09,48.434040,201675532,Brazil,NaN,NaN,46.773789,-0.595816,-0.835362,-0.561549,-0.727280,NaN,NaN,-0.516574,-0.680002,-0.516574,-0.163428
1,BRA,2016,60.8725,13.2177,5.457031,1.037554e+10,48.333340,203218114,Brazil,NaN,NaN,51.056172,-0.533596,-0.831534,-0.454299,-0.690454,NaN,NaN,-0.521016,-0.627471,-0.521016,-0.106455
2,BRA,2017,67.4713,14.1218,6.573564,1.071520e+10,50.026749,204703445,Brazil,NaN,NaN,52.345001,-0.445700,-0.830382,-0.176162,-0.633275,NaN,NaN,-0.446317,-0.521380,-0.446317,-0.075063
3,BRA,2018,70.4343,15.1538,7.688418,1.106319e+10,51.191689,206107261,Brazil,NaN,NaN,53.676862,-0.357937,-0.829192,-0.051272,-0.568008,NaN,NaN,-0.394929,-0.451602,-0.394929,-0.056673
4,BRA,2019,73.9124,15.8622,7.791551,9.392110e+09,52.624062,207455459,Brazil,NaN,NaN,45.272899,-0.349818,-0.836704,0.095329,-0.523207,NaN,NaN,-0.331745,-0.403600,-0.331745,-0.071855
5,BRA,2020,81.3427,17.4181,9.274362,5.944952e+09,54.572811,208660842,Brazil,NaN,NaN,28.490979,-0.233089,-0.851704,0.408514,-0.424807,NaN,NaN,-0.245782,-0.275271,-0.245782,-0.029489
6,BRA,2021,80.6899,19.8363,10.483914,6.350129e+09,56.830620,209550294,Brazil,NaN,NaN,30.303602,-0.137871,-0.850083,0.380999,-0.271872,NaN,NaN,-0.146186,-0.219707,-0.146186,-0.073521
7,BRA,2022,80.5278,20.9534,11.601448,7.707210e+09,60.390621,210306415,Brazil,NaN,NaN,36.647529,-0.049897,-0.844413,0.374166,-0.201223,NaN,NaN,0.010851,-0.180342,0.010851,-0.191193
8,BRA,2023,84.1506,22.4192,12.745838,8.060842e+09,NaN,211140729,Brazil,NaN,NaN,38.177579,0.040192,-0.843046,0.526866,-0.108521,NaN,NaN,NaN,-0.096127,NaN,NaN
9,BRA,2024,84.4635,NaN,12.853044,8.694668e+09,NaN,211998573,Brazil,NaN,NaN,41.012860,0.048631,-0.840511,0.540055,NaN,NaN,NaN,NaN,-0.083942,NaN,NaN


## Step 4 — Visuals + export dashboard to HTML


In [21]:
latest_year = int(df["year"].max())
latest = df[df["year"] == latest_year].copy()

fig_map = px.choropleth(
    latest,
    locations="iso3",
    color="gap_index",
    hover_name="country",
    hover_data={"demand_index": True, "supply_index": True, "gap_index": True, "iso3": False},
    title=f"Digital/AI Demand–Supply Gap Index (Latest year: {latest_year})",
)

fig_scatter = px.scatter(
    latest,
    x="supply_index",
    y="demand_index",
    text="iso3",
    hover_name="country",
    title=f"Demand vs Supply (Latest year: {latest_year})",
)

fig_trend = px.line(
    df.sort_values(["country","year"]),
    x="year",
    y=["demand_index","supply_index","gap_index"],
    color="country",
    title="Trends: Demand, Supply, Gap (2015–latest)",
)

fig_map.show()
fig_scatter.show()
fig_trend.show()

from plotly.offline import plot

html_parts = [plot(fig, include_plotlyjs='cdn', output_type='div') for fig in [fig_map, fig_scatter, fig_trend]]
html = f"""<html><head><meta charset='utf-8'><title>HW8 Dashboard</title></head>
<body><h1>HW8 — Digital/AI Demand vs Supply Dashboard</h1>
<p><b>Student:</b> Christopher Paul &nbsp;&nbsp; <b>Latest year:</b> {latest_year}</p>
{''.join(html_parts)}
</body></html>"""

with open("HW8_dashboard.html","w",encoding="utf-8") as f:
    f.write(html)

df.to_csv("HW8_demand_supply_dataset.csv", index=False)

("HW8_dashboard.html","HW8_demand_supply_dataset.csv")


('HW8_dashboard.html', 'HW8_demand_supply_dataset.csv')

## Project Summary

### Key insights
- Digital demand tends to rise fastest where connectivity, digital trade activity, and technology-oriented infrastructure are already relatively strong.
- Workforce supply lags in places where ICT employment capacity and tertiary education pipelines are comparatively thin, even when internet adoption is improving.
- The most important watch area is the “high demand / low supply” segment, where digital transformation may create skills shortages, wage pressure, and uneven access to opportunity.

### Practical recommendations
1) **Build short-cycle skills capacity:** expand employer-aligned credentials, apprenticeships, and applied training in digital, cloud, analytics, and cybersecurity fields.
2) **Strengthen enabling infrastructure:** improve broadband access, reliability, affordability, and device access to widen participation in digital work.
3) **Target workforce bottlenecks early:** use dashboard signals to identify countries or sectors where demand is rising faster than talent supply and prioritize intervention.
4) **Move toward richer labor data:** integrate job-posting or occupation-level datasets in future versions to deepen the demand-side signal and improve decision usefulness.

### Portfolio takeaway
This project demonstrates how public economic and labor data can be transformed into a decision-support dashboard for analyzing digital and AI labor-market readiness. It highlights applied skills in data acquisition, cleaning, indicator design, comparative analysis, and business-focused interpretation.
